# 第十课｜spike 为什么需要排队？

上一课一个 engine 要轮流服务很多 state，所以它不一定能在 spike 到来的同一瞬间完成所有后续工作。今天只解决：
> **当事件到达速度暂时快于消费速度时，怎样既保持顺序，又不悄悄丢事件？**

主要新概念：**有界先进先出队列（bounded First-In, First-Out queue, FIFO）**。


## 1. 概念账本

**已经知道：** spike 是离散事件；一个共享 engine 可能忙于处理其他 neuron。

**今天学习：** event queue、FIFO、full/empty，以及 backpressure。

**只预告：** 下一课才讨论一个 spike 怎样找到它的下游 synapse。


## 2. event 是什么？

这里把一个 spike 简化成一个很小的 **event（事件记录）**。最小版本只带 `source_id`：谁 spike 了。后续可以加时间、类型或其他 metadata，但本课不需要。


## 3. FIFO 保证什么？

**先进先出（First-In, First-Out, FIFO）**：最早进入队列的 event 最早离开。

如果依次进入 `2, 5, 7`，正常弹出顺序也必须是 `2, 5, 7`。FIFO 解决的是**暂存与顺序**，不是决定这些 spike 的生物学意义。


## 4. 有界队列为什么还需要 backpressure？

真实硬件 queue 容量有限。满了以后，producer 不能假装下一条 event 已经被接收。

**背压（backpressure）**：consumer/queue 用一个控制条件告诉 producer“现在不能再接收，请先保持/等待”。在本课的 Python 模型里，用 `accepted=False` 表示这件事。

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l10-flow" style="width:100%;height:auto">
<defs><marker id="l10-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="25" y="100" width="200" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="125.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">producer / pending</text>
<rect x="325" y="100" width="170" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="410.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">bounded FIFO</text>
<rect x="595" y="100" width="180" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="685.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">consumer</text>
<path d="M225 130 H325" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="270" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">push</text>
<path d="M495 130 H595" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="545" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">pop</text>
<path d="M410 100 V40 H125 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l10-flow)"/>
<text x="270" y="30" text-anchor="middle" font-family="sans-serif" font-size="14">full / retry later</text>
</svg>

在这个教学 API 里，`accepted=False` 的精确定义是：**这一次 push 没有发生**。event 仍然属于调用方，调用方不能忘掉它；等 queue 腾出空间后，应重试同一个 event。它不是“这个 spike 可以丢弃”的许可证。


## 5. Run：容量只有 3 的 queue

先预测第四个 event `9` 会怎样，再运行。

### 运行前：满、空和被拒绝的事件

**生产方（producer）**提交事件，**消费方（consumer）**取走事件。容量 3 的 FIFO 有 0 到 3 条记录；0 条为空，3 条为满。`deque.append` 加到尾部，`popleft` 从头部取出。

下面将满队列拒绝的事件保存在 `pending`。消费一个事件腾出位置后，重新提交同一个事件；只有实际接收才从 `pending` 移除。预测最终消费顺序是否仍为 `2,5,7,9`。


In [ ]:
from collections import deque

capacity = 3
q = deque()
pending = deque()
consumed = []

def show(action):
    print(f'{action:18s} queue={list(q)} full={len(q) == capacity} empty={len(q) == 0}')

for event in [2, 5, 7, 9]:
    if len(q) < capacity:
        q.append(event)
        show(f'accepted spike {event}')
    else:
        pending.append(event)
        show(f'blocked spike {event}')

while q or pending:
    if q:
        event = q.popleft()
        consumed.append(event)
        show(f'consumed spike {event}')
    if pending and len(q) < capacity:
        event = pending[0]
        q.append(event)
        pending.popleft()
        show(f'retried spike {event}')

print('consumed order:', consumed)
print('pending:', list(pending))


## 6. Observe

前三个事件先入队，第四个事件 9 保存在 `pending`。取出 2 后重新提交 9，队列变成 `[5,7,9]`。最终消费序列是 `[2,5,7,9]`，待提交队列为空。背压让提交时间推迟，事件内容和先后顺序保持。


## 7. Try It：consumer 变慢

把 capacity 改成 2。先预测哪个 push 最先被阻塞。然后尝试“push 两个 → pop 一个 → 再 push 一个”，观察 backpressure 如何解除。


## 8. 作业

打开：

[第 10 课作业：有界 FIFO 与 backpressure](../../exercises/zh/10_spike_fifo_backpressure.ipynb)

作业重点是 FIFO ordering、full/empty 和 blocked event 的 retry 语义。具体测试向量不会直接显示在作业 Notebook 中。

## 9. AI Task

让 AI 为容量 2 的 FIFO 列出一条最短操作序列，同时覆盖 empty、non-empty、full、backpressure 四种状态。要求先列预期 queue，再给代码。


## 10. Human Check

不用 AI，你应该能解释：FIFO 为什么保序；full 与 empty 各代表什么；backpressure 为什么比“满了就覆盖旧 spike”更安全；为什么 queue 只是事件运输机制，不应该偷偷改变 neuron semantics。


## 11. Engineering Handoff

本课对齐 `MOD-005 spike_fifo` 与 `IF-SPIKE-QUEUE` 的概念契约，以及 T-007 FIFO ordering / T-008 FIFO backpressure；还没有实现正式 RTL FIFO。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-010`
- Mapping: `RMD-007A / RMD-009` teaching precursor
- Module context: `MOD-005`
- Test context: `T-007 / T-008`


## 13. Exit Ticket

你能用自己的话说明 bounded FIFO 的 ordering 与 backpressure，并判断一个 full queue 为什么必须显式拒绝或延迟新 event。
